# 03b   AST Construction

**Pipeline position:** `01 -> 02 -> 03a -> **03b**`.

**Purpose:** turn the 03a structure maps into a per-document **Abstract Syntax Tree** (document -> preamble / chapters / articles -> paragraphs / tables, every node carrying a char-offset span into the source markdown) plus a **chunk list** per the doc's 03a `chunking_strategy`.

Engine: `src/ast_builder.py`   deterministic, regex only, no LLM. Spans are re-derived from the same markdown through the shared `structure_maps.find_structural_articles`, so the AST and the 03a map are always consistent.

**Step-0 lineage schema (RAG input contract):** every node and chunk carries a stable `lineage_id`:
- `{doc}:document`   `{doc}:preamble`   `{doc}:chapter:{n}`
- `{doc}:article:{n}`   `{doc}:article:{n}:para:{p}`   `{doc}:table:{seq}`
- `{doc}:sentence:{start}-{end}` (sentence-strategy docs)

**Chunk invariants (asserted below):** unique `lineage_id` per chunk; `text == source_md[span[0]:span[1]]` (stripped); no overlapping spans; thin chunks (<200 chars, e.g. a heading-only repealed placeholder) folded into a neighbour with the dropped lineage logged under `context['absorbed']`   every lineage stays resolvable to exactly one chunk.

Outputs (under `notebooks/data/ast/`):
- `{doc_id}_ast.json`   full AST (nodes, spans, children, lineage_id, metadata)
- `chunks_{doc_id}.json`   lineage_id + span + text + context per chunk
- `ast_summary.csv`   per-doc node and chunk stats (incl. thin/absorbed counts)
- `ast_handoff.json`   summary + file manifest for downstream steps

---
## 0   Setup

In [1]:
import os, sys, json, time, warnings, re
from pathlib import Path
from datetime import datetime
from collections import Counter

import pandas as pd
from rich.console import Console
from rich import print as rprint
from rich.progress import track

warnings.filterwarnings("ignore")
console = Console()

def _repo_root() -> Path:
    env = os.getenv("ENERGY_AUDIT_ROOT")
    if env and Path(env).exists():
        return Path(env).resolve()
    cur = Path.cwd().resolve()
    for cand in [cur, *cur.parents]:
        if (cand / "notebooks").is_dir() and (cand / "data").is_dir():
            return cand
    return cur.parent

ROOT = _repo_root()
sys.path.insert(0, str(ROOT / "src"))

import common as c          # paths, discovery
from structure import ast_builder as AB    # the 03b engine (this notebook just drives it)

MAPS_DIR = c.NOTEBOOKS_DATA / "structure_maps"
AST_DIR  = c.NOTEBOOKS_DATA / "ast"
AST_DIR.mkdir(parents=True, exist_ok=True)
BEST_MAP = json.loads((MAPS_DIR / "best_structure_map.json").read_text())

rprint("[bold]root :[/bold]", "../" + ROOT.name)
rprint("[bold]best map (03a)  :[/bold]", BEST_MAP["doc_id"], "(", BEST_MAP["chunking_strategy"], ")")
rprint("[bold]outputs :[/bold]", MAPS_DIR.relative_to(ROOT), "->", AST_DIR.relative_to(ROOT))


root  : <repo root>

best map (03a)  : entso_sogl_2017_1485 ( hierarchical )

outputs : notebooks/data/structure_maps -> notebooks/data/ast

---
## 1   Build all ASTs

One tree per doc. The 03a map is the structural source of truth; 03b spans come from the same markdown through the same shared regexes (03b never re-decides where an article starts).

In [2]:
t0 = time.time()
docs = {}
map_files = sorted(MAPS_DIR.glob("*_structure.json"))
rprint("[bold]%d[/bold] structure maps under %s" % (len(map_files), MAPS_DIR.relative_to(ROOT)))

for p in track(map_files, description="build asts"):
    smap = json.loads(p.read_text())
    doc_id = p.name[:-len("_structure.json")]
    md_path = c.resolve_repo_path(smap["source_md"])
    root = AB.build_ast(md_path, smap)
    (AST_DIR / f"{doc_id}_ast.json").write_text(
        json.dumps(root.to_dict(), indent=2, ensure_ascii=False), encoding="utf-8")
    docs[doc_id] = {"smap": smap, "root": root, "md_path": md_path,
                    "ast": root.to_dict()}

rprint("[bold]built %d ASTs in %.1fs[/bold]" % (len(docs), time.time() - t0))


25 structure maps under notebooks/data/structure_maps

Output()

built 25 ASTs in 1.7s

---
## 2   Build chunks + summary + hand-off

Chunks are self-contained (lineage_id + span + text + context). The summary CSV carries per-doc node/chunk stats; the hand-off records the file manifest.

In [3]:
rows = []
thin_total = absorbed_total = 0
for doc_id, d in docs.items():
    smap, root = d["smap"], d["root"]
    content = d["md_path"].read_text(encoding="utf-8")
    chunks = AB.build_chunks(content, root, smap["chunking_strategy"])
    (AST_DIR / f"chunks_{doc_id}.json").write_text(
        json.dumps([ch.to_dict() for ch in chunks], indent=1), encoding="utf-8")
    lens = [len(x.text) for x in chunks]
    thin = sum(1 for x in chunks if len(x.text) < 200)
    absorbed = sum(len(x.context.get("absorbed") or []) for x in chunks)
    thin_total += thin
    absorbed_total += absorbed
    rows.append({
        "doc_id": doc_id,
        "category": smap["category"],
        "strategy": smap["chunking_strategy"],
        "articles": root.count_of("article"),
        "paragraphs": root.count_of("paragraph"),
        "tables": root.count_of("table"),
        "chapters": len([n for n in root.children if n.node_type == "chapter"]),
        "chunks": len(chunks),
        "thin_chunks": thin,
        "absorbed_lineages": absorbed,
        "chunk_chars_min": min(lens) if lens else 0,
        "chunk_chars_mean": int(sum(lens) / max(len(lens), 1)) if lens else 0,
        "chunk_chars_max": max(lens) if lens else 0,
    })

df = pd.DataFrame(rows).sort_values(["strategy", "doc_id"])
df.to_csv(AST_DIR / "ast_summary.csv", index=False)
rprint("[bold]chunk sizes by strategy:[/bold]")
for s, g in df.groupby("strategy"):
    rprint("  %-22s n=%-3d mean=%-6d max=%-6d thin=%d absorbed=%d" % (
        s, len(g), int(g["chunk_chars_mean"].mean()), int(g["chunk_chars_max"].max()),
        int(g["thin_chunks"].sum()), int(g["absorbed_lineages"].sum())))

handoff = {
    "generated": datetime.now().isoformat(),
    "docs": len(docs),
    "strategy_mix": {s: int(n) for s, n in Counter(r["strategy"] for r in rows).items()},
    "total_articles": int(df["articles"].sum()),
    "total_paragraphs": int(df["paragraphs"].sum()),
    "total_table_nodes": int(df["tables"].sum()),
    "total_chunks": int(df["chunks"].sum()),
    "thin_chunks": int(thin_total),
    "absorbed_lineages": int(absorbed_total),
    "lineage_schema": "{doc}:document|preamble|chapter:{n}|article:{n}|article:{n}:para:{p}|table:{seq}|sentence:{start}-{end}",
    "best_demo_doc": BEST_MAP["doc_id"],
    "files": {
        "per_doc_ast": "notebooks/data/ast/{doc_id}_ast.json (lineage_id on every node)",
        "chunks": "notebooks/data/ast/chunks_{doc_id}.json (lineage_id, span, text, context)",
        "summary": "notebooks/data/ast/ast_summary.csv",
        "source_md": "each structure map's source_md field (repo-relative)",
    },
    "note": "chunk invariants: unique lineage_id; text == source[span[0]:span[1]].strip(); "
            "no overlapping spans; thin chunks folded (context.absorbed records dropped "
            "lineages, each of which resolves to exactly one chunk)",
}
(AST_DIR / "ast_handoff.json").write_text(json.dumps(handoff, indent=2))

rprint("[bold]outputs under ast/:[/bold]")
for f in sorted(AST_DIR.iterdir()):
    rprint("   ", f.name, "%10d" % len(f.read_bytes()), "bytes")


chunk sizes by strategy:

article_based          n=11  mean=5020   max=161504 thin=0 absorbed=6

hierarchical           n=10  mean=3664   max=246616 thin=0 absorbed=4

table_then_sentence    n=4   mean=241    max=9566   thin=1473 absorbed=8

outputs under ast/:

...

metering_data_2023_1162_ast.json      34907 bytes

mica_2023_1114_ast.json     374981 bytes

nis2_dir_2022_2555_ast.json     172509 bytes

remit_1227_2011_ast.json      52759 bytes

remit_ii_2024_1106_ast.json     100930 bytes

---
## 3   Validate

Consistency invariants between the 03a map and the 03b AST (same regexes, same markdown, all must hold exactly):
- article-node count == map `article_count`
- article spans do not overlap and stay in `0..len(content)`
- no degenerate/blank node spans for article/paragraph/table
- every paragraph sits inside its parent article's span
- every structural node carries a `lineage_id`

Chunk invariants (Step 0):
- every chunk has a unique `lineage_id`
- `chunk.text == source[span[0]:span[1]].strip()` (source-faithful slice)
- chunk spans within a doc never overlap
- no unresolved thin (<200 char) chunks (only `thin`-tagged survivors may remain)

In [4]:
def _walk(node, acc):
    acc.append(node)
    for ch in node["children"]:
        _walk(ch, acc)
    return acc

problems = []
for doc_id, d in docs.items():
    smap, ast = d["smap"], d["ast"]
    content = d["md_path"].read_text(encoding="utf-8")
    all_nodes = _walk(ast, [])
    articles = [n for n in all_nodes if n["type"] == "article"]
    if len(articles) != smap["article_count"]:
        problems.append((doc_id, "article count %d != map %d" % (len(articles), smap["article_count"])))
    spans = sorted((a["span"][0], a["span"][1], a["identifier"]) for a in articles)
    for i in range(len(spans) - 1):
        if spans[i][1] > spans[i + 1][0]:
            problems.append((doc_id, "overlap %s | %s" % (spans[i][2], spans[i + 1][2])))
    for s in spans:
        if s[0] < 0 or s[1] > len(content):
            problems.append((doc_id, "out of range %s span=%s" % (s[2], s[1:])))
    for n in all_nodes:
        s, e = n["span"]
        if n["type"] in ("article", "paragraph", "table") and (e - s) < 8:
            problems.append((doc_id, "degenerate span %s %s" % (n["identifier"], (s, e))))
        if (e - s) > 0 and content[s:e].strip() == "":
            problems.append((doc_id, "blank span %s" % n["identifier"]))
    art = {a["identifier"]: a for a in articles}
    for n in all_nodes:
        if n["type"] == "paragraph" and n["identifier"].startswith("P-"):
            parent = n["identifier"].split(".")[0]
            if parent in art and not (art[parent]["span"][0] <= n["span"][0] < art[parent]["span"][1]):
                problems.append((doc_id, "paragraph outside parent %s" % n["identifier"]))
    for n in all_nodes:
        if n["type"] in ("article", "preamble", "table", "paragraph"):
            if not n.get("lineage_id"):
                problems.append((doc_id, "missing lineage on %s %s" % (n["type"], n["identifier"])))

# --- chunk-level invariants (Step 0) ---
chunk_problems = []
for f in sorted(AST_DIR.glob("chunks_*.json")):
    doc_id = f.name.replace("chunks_", "", 1).replace(".json", "")
    content = docs[doc_id]["md_path"].read_text(encoding="utf-8")
    chs = json.loads(f.read_text())
    lins = [x["lineage_id"] for x in chs]
    if len(lins) != len(set(lins)):
        chunk_problems.append((doc_id, "duplicate lineage_id"))
    ss = sorted((x["span"][0], x["span"][1]) for x in chs)
    for i in range(len(ss) - 1):
        if ss[i][1] > ss[i + 1][0]:
            chunk_problems.append((doc_id, "chunk spans overlap %s > %s" % (ss[i], ss[i + 1])))
    for x in chs:
        s, e = x["span"]
        if content[s:e].strip() != x["text"]:
            chunk_problems.append((doc_id, "text != source slice for %s" % x["lineage_id"]))
        if len(x["text"]) < 200 and not x.get("context", {}).get("thin"):
            chunk_problems.append((doc_id, "unresolved thin chunk %s (%d ch)" % (x["lineage_id"], len(x["text"]))))

if problems or chunk_problems:
    rprint("[bold red]VALIDATION FAILURES:[/bold red]")
    for doc_id, msg in (problems + chunk_problems)[:30]:
        rprint("  ", doc_id, "-", msg)
    raise AssertionError("%d problems (ast=%d chunk=%d)" % (
        len(problems) + len(chunk_problems), len(problems), len(chunk_problems)))

rprint("[bold green]all %d ASTs validate clean (counts, overlaps, spans, nesting, lineage).[/bold green]" % len(docs))
rprint("[bold green]chunk invariants hold for all docs (unique lineage, source slice, no overlap, no unresolved thin).[/bold green]")


all 25 ASTs validate clean (counts, overlaps, spans, nesting, lineage).

chunk invariants hold for all docs (unique lineage, source slice, no overlap, no unresolved thin).

---
## 4   Spot-check the best doc (from 03a)

Skeleton of the best-coverage map's tree (with lineage IDs) + its first chunks, so span quality can be eyeballed before downstream steps rely on it.

In [5]:
best = BEST_MAP["doc_id"]
d = docs[best]
ast, content = d["ast"], d["md_path"].read_text(encoding="utf-8")

def _print_node(n, depth=0):
    if depth > 2:
        return
    label = "%s %s" % (n["type"], n["identifier"])
    if n.get("lineage_id"):
        label += "  [%s]" % n["lineage_id"]
    if n.get("title"):
        label += " - " + n["title"][:36]
    print("    " * depth + label)
    for ch in n["children"][:10]:
        _print_node(ch, depth + 1)
    if len(n["children"]) > 10:
        print("    " * (depth + 1) + "... %d more" % (len(n["children"]) - 10))

rprint("[bold]%s[/bold]  (strategy = %s)" % (best, ast["metadata"]["chunking_strategy"]))
_print_node(ast)

chunks = json.loads((AST_DIR / f"chunks_{best}.json").read_text())
rprint("\n[bold]first 3 chunks (%s total):[/bold]" % len(chunks))
for ch in chunks[:3]:
    rprint("  ", ch["lineage_id"], ch["node_type"], "%6d" % len(ch["text"]), "chars |", " ".join(ch["text"].split())[:64])

ep = docs["eprivacy_dir_2002_58"]["md_path"].read_text(encoding="utf-8")
epc = json.loads((AST_DIR / "chunks_eprivacy_dir_2002_58.json").read_text())
rprint("\n[bold]shell-folding demo   eprivacy (heading-only repealed articles in source):[/bold]")
for x in epc:
    if x.get("context", {}).get("absorbed"):
        rprint("   %s  absorbs %s" % (x["lineage_id"], x["context"]["absorbed"]))


entso_sogl_2017_1485  (strategy = hierarchical)

document entso_sogl_2017_1485  [entso_sogl_2017_1485:document] - COMMISSION REGULATION (EU) 2017/1485
    preamble entso_sogl_2017_1485:preamble  [entso_sogl_2017_1485:preamble]
    chapter CHP-I  [entso_sogl_2017_1485:chapter:I] - ## **GENERAL PROVISIONS** _Article 1
    chapter CHP-II  [entso_sogl_2017_1485:chapter:II] - ## **OPERATIONAL SECURITY** ## TITLE
    chapter CHP-1  [entso_sogl_2017_1485:chapter:1] - ## **OPERATIONAL SECURITY REQUIREMEN
    chapter CHP-2  [entso_sogl_2017_1485:chapter:2] - ## **DATA EXCHANGE** _CHAPTER 1_ ## 
    chapter CHP-3  [entso_sogl_2017_1485:chapter:3] - ## **COMPLIANCE** _CHAPTER 1_ ## _**
    chapter CHP-4  [entso_sogl_2017_1485:chapter:4] - **TRAINING** _Article 58_ ## **Train
    chapter CHP-III  [entso_sogl_2017_1485:chapter:III] - ## **OPERATIONAL PLANNING** ## TITLE
    chapter CHP-1  [entso_sogl_2017_1485:chapter:1] - ## **DATA FOR OPERATIONAL SECURITY A
    chapter CHP-2  [entso_sogl_2017_1485:chapter:2] - ## **OPERATIONAL SECURITY ANALYSIS

first 3 chunks (190 total):

entso_sogl_2017_1485:preamble preamble   8755 chars | Official Journal of the European Union L 220/1 25.8.2017 
EN ## I

entso_sogl_2017_1485:article:1 article    804 chars | _Article 1_ ## **Subject matter** For the purpose of 
safeguardin

entso_sogl_2017_1485:article:2 article   3263 chars | ## _Article 2_ ## **Scope** 1. The rules and requirements 
set ou

shell-folding demo   eprivacy (heading-only repealed articles in source):

eprivacy_dir_2002_58:article:7  absorbs ['eprivacy_dir_2002_58:article:6']

eprivacy_dir_2002_58:article:16  absorbs ['eprivacy_dir_2002_58:article:15']

eprivacy_dir_2002_58:article:19  absorbs ['eprivacy_dir_2002_58:article:20']